# Transformer Grad-CAM visualization + comparison against original single-view ALBEF

This notebook generates **two PDFs**:

1. **Transformer-only PDF**
   - One row per case:
     **Original + GT | Transformer heatmap | Transformer overlay + GT**

2. **Transformer vs original single-view ALBEF PDF**
   - One row per case:
     **Original + GT | Transformer overlay + GT | Original single-view ALBEF overlay + GT**

The notebook is flexible:
- If you already have a canonical `image_id,label` CSV, provide it.
- Otherwise it can infer available cases from the saved transformer heatmap files.

Panel titles are truthful:
- `GT box` if a matching VinDr bbox exists.
- `no GT box` if no matching bbox row is found.


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

pd.set_option("display.max_columns", 100)


## Configuration


In [ ]:
TRANSFORMER_HEATMAPS_DIR = Path(
    "/CHANGE/THIS/TO/YOUR/TRANSFORMER_GRADCAM_DIR"
)

ORIGINAL_ALBEF_HEATMAPS_DIR = Path(
    "/CHANGE/THIS/TO/YOUR/ORIGINAL_VIEW_ALBEF_GRADCAM_DIR"
)

CANONICAL_CASES_CSV = None
# Example:
# CANONICAL_CASES_CSV = Path(
#     "/home/vault/iwi5/iwi5362h/results/visualization/"
#     "chexzero_albef_biovil_comparison/canonical_100_image_label_pair_order.csv"
# )

TARGET_LABELS = ["Cardiomegaly", "Pleural effusion"]

IMAGES_ROOT = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test"
)
ANNOTATIONS_CSV = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv"
)
IMAGE_METADATA_CSV = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv"
)

OUTPUT_DIR = Path(
    "/home/vault/iwi5/iwi5362h/results/visualization/transformer_vs_original_albef"
)

CASES_PER_PAGE = 5
OVERLAY_ALPHA = 0.50
CMAP_NAME = "magma"
FIG_DPI = 150
PDF_DPI = 150
SAVE_PNG_PAGES = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [IMAGES_ROOT, ANNOTATIONS_CSV, IMAGE_METADATA_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Transformer heatmaps:", TRANSFORMER_HEATMAPS_DIR)
print("Original ALBEF:     ", ORIGINAL_ALBEF_HEATMAPS_DIR)
print("Output dir:         ", OUTPUT_DIR)


## Shared helpers


In [ ]:
def safe_torch_load(path):
    path = Path(path)
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_2d_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    arr = np.asarray(value, dtype=np.float32).squeeze()
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError("Heatmap contains NaN or infinity")
    return arr


def minmax_for_display(arr):
    arr = as_2d_numpy(arr)
    lo = float(arr.min())
    hi = float(arr.max())
    if hi - lo <= 1e-12:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - lo) / (hi - lo)).astype(np.float32)


def ensure_display_range(arr):
    arr = as_2d_numpy(arr)
    if float(arr.min()) >= -1e-6 and float(arr.max()) <= 1.0 + 1e-6:
        return np.clip(arr, 0, 1).astype(np.float32)
    return minmax_for_display(arr)


def resize_map(arr, size_wh):
    arr = ensure_display_range(arr)
    if arr.shape == (size_wh[1], size_wh[0]):
        return arr
    pil = Image.fromarray(np.round(arr * 255).astype(np.uint8), mode="L")
    pil = pil.resize(size_wh, Image.Resampling.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255.0


def make_overlay(image, heatmap, alpha=OVERLAY_ALPHA):
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    hm = resize_map(heatmap, image.size)
    color = colormaps[CMAP_NAME](np.clip(hm, 0, 1))[..., :3]
    alpha_map = (alpha * np.clip(hm, 0, 1))[..., None]
    return np.clip((1 - alpha_map) * rgb + alpha_map * color, 0, 1)


def sanitize_filename(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip()).strip("_")


def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(
        f"Could not find {purpose}. Available columns: {list(df.columns)}"
    )


def load_image(image_id):
    path = IMAGES_ROOT / f"{image_id}.png"
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as handle:
        return handle.convert("RGB")


def label_slug(label):
    return re.sub(r"[^a-z0-9]+", "_", str(label).lower()).strip("_")


## Load VinDr annotations and original image dimensions


In [ ]:
ann_raw = pd.read_csv(ANNOTATIONS_CSV)
ann_map = {
    "image_id": choose_column(
        ann_raw, ["image_id", "imageid"], "annotation image ID"
    ),
    "class_name": choose_column(
        ann_raw, ["class_name", "class", "label"], "annotation label"
    ),
    "x_min": choose_column(ann_raw, ["x_min", "xmin", "x1"], "x_min"),
    "y_min": choose_column(ann_raw, ["y_min", "ymin", "y1"], "y_min"),
    "x_max": choose_column(ann_raw, ["x_max", "xmax", "x2"], "x_max"),
    "y_max": choose_column(ann_raw, ["y_max", "ymax", "y2"], "y_max"),
}
annotations_df = ann_raw[list(ann_map.values())].rename(
    columns={v: k for k, v in ann_map.items()}
)
annotations_df["image_id"] = annotations_df["image_id"].astype(str)

meta_raw = pd.read_csv(IMAGE_METADATA_CSV)
meta_map = {
    "image_id": choose_column(
        meta_raw, ["image_id", "imageid"], "metadata image ID"
    ),
    "width": choose_column(
        meta_raw,
        ["width", "image_width", "original_width", "dim1", "w"],
        "original width",
    ),
    "height": choose_column(
        meta_raw,
        ["height", "image_height", "original_height", "dim0", "h"],
        "original height",
    ),
}
metadata_df = meta_raw[list(meta_map.values())].rename(
    columns={v: k for k, v in meta_map.items()}
)
metadata_df["image_id"] = metadata_df["image_id"].astype(str)

if metadata_df["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs detected in metadata")

metadata_lookup = (
    metadata_df.set_index("image_id")[["width", "height"]].to_dict("index")
)


def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[
        (annotations_df.image_id == str(image_id))
        & (annotations_df.class_name == str(label))
    ]
    if subset.empty:
        return []

    if str(image_id) not in metadata_lookup:
        raise KeyError(f"Missing original dimensions for {image_id}")

    original = metadata_lookup[str(image_id)]
    sx = displayed_size[0] / float(original["width"])
    sy = displayed_size[1] / float(original["height"])

    boxes = []
    for row in subset.itertuples(index=False):
        x1 = np.clip(float(row.x_min) * sx, 0, displayed_size[0])
        y1 = np.clip(float(row.y_min) * sy, 0, displayed_size[1])
        x2 = np.clip(float(row.x_max) * sx, 0, displayed_size[0])
        y2 = np.clip(float(row.y_max) * sy, 0, displayed_size[1])
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return boxes


def draw_boxes(ax, boxes, color="lime", linewidth=2):
    for x1, y1, x2, y2 in boxes:
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor=color,
                linewidth=linewidth,
            )
        )


def gt_status_text(boxes):
    return "GT box" if len(boxes) > 0 else "no GT box"


print(
    f"Loaded {len(annotations_df):,} annotation rows and "
    f"{len(metadata_df):,} metadata rows"
)


## Heatmap-file discovery and loading


In [ ]:
MAP_KEY_PRIORITY = (
    "cam_vis_up",
    "gradcam_vis_up",
    "cam_vis",
    "gradcam_vis",
    "cam_positive_raw",
    "cam_raw",
    "heatmap_vis",
    "heatmap",
    "gradcam",
    "cam",
    "map",
)


def infer_image_id_from_payload_or_path(payload, path):
    if isinstance(payload, dict):
        if "__metadata__" in payload and isinstance(payload["__metadata__"], dict):
            image_id = payload["__metadata__"].get("image_id")
            if image_id is not None and str(image_id).strip():
                return str(image_id)

        for key in ("image_id", "id"):
            if key in payload and str(payload[key]).strip():
                return str(payload[key])

    stem = Path(path).stem
    if "__" in stem:
        return stem.split("__", 1)[0]
    return stem


def extract_map_from_object(obj, context):
    if torch.is_tensor(obj) or isinstance(obj, np.ndarray):
        return ensure_display_range(obj), "<direct tensor>"

    if isinstance(obj, dict):
        for key in MAP_KEY_PRIORITY:
            if key in obj:
                return ensure_display_range(obj[key]), key

        candidates = []
        for key, value in obj.items():
            try:
                arr = ensure_display_range(value)
                candidates.append((key, arr))
            except Exception:
                pass

        if len(candidates) == 1:
            key, arr = candidates[0]
            warnings.warn(
                f"{context}: using sole 2D field {key!r}. "
                "If intentional, add it to MAP_KEY_PRIORITY."
            )
            return arr, key

        raise KeyError(
            f"{context}: no recognized map field. "
            f"available keys={list(obj.keys())}; "
            f"2D candidates={[k for k, _ in candidates]}"
        )

    raise TypeError(
        f"{context}: unsupported object type {type(obj)}"
    )


def find_heatmap_file(root_dir, image_id, label):
    root_dir = Path(root_dir)
    slug = label_slug(label)

    preferred = [
        root_dir / f"{image_id}.pt",
        root_dir / "maps" / f"{image_id}.pt",
        root_dir / "heatmaps" / f"{image_id}.pt",
    ]
    for path in preferred:
        if path.is_file():
            return path

    exact = list(root_dir.rglob(f"{image_id}.pt"))
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        good = [p for p in exact if ("maps" in p.parts or "heatmaps" in p.parts)]
        if len(good) == 1:
            return good[0]
        raise RuntimeError(
            f"Ambiguous files for image_id={image_id}: {exact[:10]}"
        )

    fuzzy = list(root_dir.rglob(f"{image_id}*.pt"))
    label_matches = [p for p in fuzzy if slug in p.stem.lower()]
    if len(label_matches) == 1:
        return label_matches[0]
    if len(fuzzy) == 1:
        return fuzzy[0]

    raise FileNotFoundError(
        f"No unambiguous heatmap file for image_id={image_id}, label={label!r} "
        f"under {root_dir}"
    )


def load_method_map(root_dir, image_id, label):
    path = find_heatmap_file(root_dir, image_id, label)
    payload = safe_torch_load(path)

    obj = payload
    if isinstance(payload, dict):
        if label in payload:
            obj = payload[label]
        elif "heatmaps" in payload and isinstance(payload["heatmaps"], dict):
            if label in payload["heatmaps"]:
                obj = payload["heatmaps"][label]

    arr, used_key = extract_map_from_object(
        obj,
        f"{path}:{label}",
    )
    return arr, payload, path, used_key


## Build the case list


In [ ]:
def case_sort_key(df):
    label_order = {label: i for i, label in enumerate(TARGET_LABELS)}
    ordered = df.copy()
    ordered["_label_order"] = ordered["label"].map(
        lambda x: label_order.get(x, len(label_order))
    )
    ordered = ordered.sort_values(
        ["_label_order", "image_id"],
        kind="stable",
    )
    return ordered.drop(columns="_label_order").reset_index(drop=True)


def load_cases_from_canonical_csv(path):
    df = pd.read_csv(path)
    image_col = choose_column(df, ["image_id", "imageid"], "image_id")
    label_col = choose_column(df, ["label", "class_name", "class"], "label")
    out = df[[image_col, label_col]].rename(
        columns={image_col: "image_id", label_col: "label"}
    )
    out["image_id"] = out["image_id"].astype(str)
    out["label"] = out["label"].astype(str)
    out = out[out["label"].isin(TARGET_LABELS)].copy()

    if out.empty:
        raise ValueError(
            f"{path}: no rows remain after filtering to TARGET_LABELS={TARGET_LABELS}"
        )

    if out.duplicated(["image_id", "label"]).any():
        raise ValueError(f"{path}: duplicate image-label pairs found")

    return case_sort_key(out)


def discover_cases_from_transformer_outputs(root_dir):
    root_dir = Path(root_dir)
    pt_files = sorted(root_dir.rglob("*.pt"))

    if not pt_files:
        raise RuntimeError(f"No .pt files found under {root_dir}")

    records = []
    for path in pt_files:
        payload = safe_torch_load(path)
        image_id = infer_image_id_from_payload_or_path(payload, path)

        candidate_labels = []
        if isinstance(payload, dict):
            candidate_labels.extend(
                [label for label in TARGET_LABELS if label in payload]
            )
            if "heatmaps" in payload and isinstance(payload["heatmaps"], dict):
                candidate_labels.extend(
                    [label for label in TARGET_LABELS if label in payload["heatmaps"]]
                )

        candidate_labels = list(dict.fromkeys(candidate_labels))

        if not candidate_labels:
            for label in TARGET_LABELS:
                if label_slug(label) in path.stem.lower():
                    candidate_labels.append(label)

        for label in candidate_labels:
            try:
                _map, _payload, _resolved, _key = load_method_map(
                    root_dir, image_id, label
                )
                records.append({"image_id": str(image_id), "label": str(label)})
            except Exception:
                pass

    if not records:
        raise RuntimeError(
            "Could not infer any cases from transformer heatmaps. "
            "Provide CANONICAL_CASES_CSV instead."
        )

    out = pd.DataFrame(records).drop_duplicates(
        ["image_id", "label"]
    ).reset_index(drop=True)

    return case_sort_key(out)


if CANONICAL_CASES_CSV is not None:
    if not Path(CANONICAL_CASES_CSV).is_file():
        raise FileNotFoundError(CANONICAL_CASES_CSV)
    cases_df = load_cases_from_canonical_csv(CANONICAL_CASES_CSV)
    case_source = f"canonical CSV: {CANONICAL_CASES_CSV}"
else:
    if not TRANSFORMER_HEATMAPS_DIR.exists():
        raise FileNotFoundError(
            "Set TRANSFORMER_HEATMAPS_DIR before inferring cases"
        )
    cases_df = discover_cases_from_transformer_outputs(
        TRANSFORMER_HEATMAPS_DIR
    )
    case_source = f"inferred from transformer heatmaps: {TRANSFORMER_HEATMAPS_DIR}"

print("Case source:", case_source)
print("Image-label pairs:", len(cases_df))
print("Unique images:    ", cases_df["image_id"].nunique())
display(cases_df["label"].value_counts().rename("pairs").to_frame())
display(cases_df.head(10))


## Validate that both methods cover the same cases


In [ ]:
if not TRANSFORMER_HEATMAPS_DIR.exists():
    raise FileNotFoundError(TRANSFORMER_HEATMAPS_DIR)

if not ORIGINAL_ALBEF_HEATMAPS_DIR.exists():
    raise FileNotFoundError(ORIGINAL_ALBEF_HEATMAPS_DIR)

validation_rows = []
errors = []

for row in cases_df.itertuples(index=False):
    image_id = str(row.image_id)
    label = str(row.label)

    try:
        t_map, _, t_path, t_key = load_method_map(
            TRANSFORMER_HEATMAPS_DIR, image_id, label
        )
    except Exception as exc:
        errors.append(
            {
                "method": "transformer",
                "image_id": image_id,
                "label": label,
                "error": repr(exc),
            }
        )
        continue

    try:
        o_map, _, o_path, o_key = load_method_map(
            ORIGINAL_ALBEF_HEATMAPS_DIR, image_id, label
        )
    except Exception as exc:
        errors.append(
            {
                "method": "original_albef",
                "image_id": image_id,
                "label": label,
                "error": repr(exc),
            }
        )
        continue

    validation_rows.append(
        {
            "image_id": image_id,
            "label": label,
            "transformer_path": str(t_path),
            "transformer_key": t_key,
            "transformer_shape": str(tuple(t_map.shape)),
            "original_path": str(o_path),
            "original_key": o_key,
            "original_shape": str(tuple(o_map.shape)),
        }
    )

validation_df = pd.DataFrame(validation_rows)
errors_df = pd.DataFrame(errors)

print("Validated pairs:", len(validation_df))
print("Errors:         ", len(errors_df))

if not errors_df.empty:
    display(errors_df.head(20))
    raise RuntimeError(
        "Some requested cases are missing or invalid. "
        "Fix the paths or case list before rendering the PDFs."
    )

display(validation_df.head())


## Preview one case


In [ ]:
CASE_INDEX = 0

row = cases_df.iloc[CASE_INDEX]
image = load_image(row.image_id)
boxes = get_boxes(row.image_id, row.label, image.size)

t_map, t_payload, t_path, t_key = load_method_map(
    TRANSFORMER_HEATMAPS_DIR, row.image_id, row.label
)
o_map, o_payload, o_path, o_key = load_method_map(
    ORIGINAL_ALBEF_HEATMAPS_DIR, row.image_id, row.label
)

t_up = resize_map(t_map, image.size)
o_up = resize_map(o_map, image.size)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=FIG_DPI)

axes[0].imshow(image)
draw_boxes(axes[0], boxes)
axes[0].set_title(
    f"{CASE_INDEX+1:03d}. {row.image_id}\n"
    f"{row.label} | {gt_status_text(boxes)}"
)

axes[1].imshow(t_up, cmap=CMAP_NAME, vmin=0, vmax=1)
axes[1].set_title(
    f"Transformer map\nfield={t_key}\nnative={tuple(t_map.shape)}"
)

axes[2].imshow(make_overlay(image, t_up))
draw_boxes(axes[2], boxes)
axes[2].set_title("Transformer overlay + GT")

axes[3].imshow(make_overlay(image, o_up))
draw_boxes(axes[3], boxes)
axes[3].set_title(
    f"Original ALBEF overlay + GT\nfield={o_key}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Transformer file:", t_path)
print("Original ALBEF file:", o_path)


## Render PDF 1: transformer-only Grad-CAMs


In [ ]:
transformer_pdf_path = OUTPUT_DIR / "transformer_gradcams_all_cases.pdf"
transformer_png_dir = OUTPUT_DIR / "transformer_pages"
transformer_png_dir.mkdir(parents=True, exist_ok=True)


def create_transformer_page(page_df, page_number, start_index):
    nrows = len(page_df)
    fig, axes = plt.subplots(
        nrows,
        3,
        figsize=(12.5, 3.45 * nrows),
        dpi=FIG_DPI,
        squeeze=False,
    )

    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        gt_text = gt_status_text(boxes)

        t_map, _, _, t_key = load_method_map(
            TRANSFORMER_HEATMAPS_DIR,
            row.image_id,
            row.label,
        )
        t_up = resize_map(t_map, image.size)
        case_number = start_index + r + 1

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes)
        axes[r, 0].set_title(
            f"{case_number:03d}. {row.image_id}\n"
            f"{row.label} | {gt_text}",
            fontsize=8,
        )

        axes[r, 1].imshow(t_up, cmap=CMAP_NAME, vmin=0, vmax=1)
        axes[r, 1].set_title(
            f"Transformer heatmap\nfield={t_key} | native={tuple(t_map.shape)}",
            fontsize=8,
        )

        axes[r, 2].imshow(make_overlay(image, t_up))
        draw_boxes(axes[r, 2], boxes)
        axes[r, 2].set_title(
            "Transformer overlay + GT",
            fontsize=8,
        )

        for ax in axes[r]:
            ax.axis("off")

    fig.suptitle(
        f"Transformer Grad-CAMs - page {page_number:02d}",
        fontsize=14,
        fontweight="bold",
        y=1.002,
    )
    plt.tight_layout()
    return fig


num_pages = math.ceil(len(cases_df) / CASES_PER_PAGE)
print(f"Rendering transformer-only PDF with {len(cases_df)} pairs across {num_pages} pages")

with PdfPages(transformer_pdf_path) as pdf:
    for page_index in range(num_pages):
        start = page_index * CASES_PER_PAGE
        page_df = cases_df.iloc[start : start + CASES_PER_PAGE]

        fig = create_transformer_page(
            page_df,
            page_index + 1,
            start,
        )

        pdf.savefig(fig, bbox_inches="tight", dpi=PDF_DPI)

        if SAVE_PNG_PAGES:
            fig.savefig(
                transformer_png_dir / f"page_{page_index+1:02d}.png",
                dpi=FIG_DPI,
                bbox_inches="tight",
            )

        plt.close(fig)

print("Saved:", transformer_pdf_path)


## Render PDF 2: transformer vs original single-view ALBEF


In [ ]:
comparison_pdf_path = OUTPUT_DIR / "transformer_vs_original_albef_all_cases.pdf"
comparison_png_dir = OUTPUT_DIR / "comparison_pages"
comparison_png_dir.mkdir(parents=True, exist_ok=True)


def create_comparison_page(page_df, page_number, start_index):
    nrows = len(page_df)
    fig, axes = plt.subplots(
        nrows,
        3,
        figsize=(13.5, 3.55 * nrows),
        dpi=FIG_DPI,
        squeeze=False,
    )

    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        gt_text = gt_status_text(boxes)

        t_map, _, _, t_key = load_method_map(
            TRANSFORMER_HEATMAPS_DIR,
            row.image_id,
            row.label,
        )
        o_map, _, _, o_key = load_method_map(
            ORIGINAL_ALBEF_HEATMAPS_DIR,
            row.image_id,
            row.label,
        )

        t_up = resize_map(t_map, image.size)
        o_up = resize_map(o_map, image.size)
        case_number = start_index + r + 1

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes)
        axes[r, 0].set_title(
            f"{case_number:03d}. {row.image_id}\n"
            f"{row.label} | {gt_text}",
            fontsize=8,
        )

        axes[r, 1].imshow(make_overlay(image, t_up))
        draw_boxes(axes[r, 1], boxes)
        axes[r, 1].set_title(
            f"Transformer overlay + GT\nfield={t_key}",
            fontsize=8,
        )

        axes[r, 2].imshow(make_overlay(image, o_up))
        draw_boxes(axes[r, 2], boxes)
        axes[r, 2].set_title(
            f"Original single-view ALBEF + GT\nfield={o_key}",
            fontsize=8,
        )

        for ax in axes[r]:
            ax.axis("off")

    fig.suptitle(
        f"Transformer vs original single-view ALBEF - page {page_number:02d}",
        fontsize=14,
        fontweight="bold",
        y=1.002,
    )
    plt.tight_layout()
    return fig


num_pages = math.ceil(len(cases_df) / CASES_PER_PAGE)
print(f"Rendering comparison PDF with {len(cases_df)} pairs across {num_pages} pages")

with PdfPages(comparison_pdf_path) as pdf:
    for page_index in range(num_pages):
        start = page_index * CASES_PER_PAGE
        page_df = cases_df.iloc[start : start + CASES_PER_PAGE]

        fig = create_comparison_page(
            page_df,
            page_index + 1,
            start,
        )

        pdf.savefig(fig, bbox_inches="tight", dpi=PDF_DPI)

        if SAVE_PNG_PAGES:
            fig.savefig(
                comparison_png_dir / f"page_{page_index+1:02d}.png",
                dpi=FIG_DPI,
                bbox_inches="tight",
            )

        plt.close(fig)

print("Saved:", comparison_pdf_path)


## Save the case order and output summary


In [ ]:
case_order_path = OUTPUT_DIR / "case_order.csv"
cases_df.to_csv(case_order_path, index=False)

summary_df = pd.DataFrame(
    [
        {
            "output_type": "transformer_only_pdf",
            "path": str(transformer_pdf_path),
            "num_pairs": int(len(cases_df)),
            "num_unique_images": int(cases_df["image_id"].nunique()),
        },
        {
            "output_type": "transformer_vs_original_pdf",
            "path": str(comparison_pdf_path),
            "num_pairs": int(len(cases_df)),
            "num_unique_images": int(cases_df["image_id"].nunique()),
        },
    ]
)

summary_path = OUTPUT_DIR / "pdf_outputs.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved case order:", case_order_path)
print("Saved summary:   ", summary_path)
display(summary_df)


## Notes

- The comparison unit is an **image-label pair**, not only an image. One CXR can appear once for Cardiomegaly and once for Pleural effusion.
- The notebook writes `GT box` only when a matching VinDr bbox is found in `annotations_test.csv`. Otherwise it says `no GT box`.
- The loader first tries fields such as `cam_vis_up` and `cam_vis`. If your saved payload uses a different field name, add it to `MAP_KEY_PRIORITY`.
- If you want a fixed 100-case order, set `CANONICAL_CASES_CSV` to a saved CSV with columns `image_id,label`.
